# 01 — Fine-tuning de BGE-M3 sur le vocabulaire IFRS 17 (D-046)

**Problème mesuré.** Sur le quiz du cours (15 questions écrites par un enseignant), la recherche hybride
retrouve le bon paragraphe dans les 5 premiers résultats pour **26,7 %** des questions, contre **83,3 %**
sur notre jeu principal (D-044). La réécriture de la question par le LLM n'y change rien (D-045, F-012) :
le modèle d'embedding ne relie pas « maille » au §16 ni « option OCI » au §88.

**Méthode.** Apprentissage contrastif sur ~1 500 paires (question générée → passage de la norme), avec
un négatif difficile (passage proche par les mots, mais sans la réponse) et les autres passages du lot
comme négatifs (*MultipleNegativesRankingLoss*, version *Cached* pour tenir dans un T4).

**Règle d'évaluation.** On ne mesure que sur des questions **écrites par des humains et jamais vues à
l'entraînement** : le quiz du cours et nos 56 questions. Les questions générées trop proches d'une
question de test ont déjà été retirées par `rag/finetune.py`.

| Critère | Go | No-go |
| --- | --- | --- |
| Quiz du cours, Recall@5 (dense) | hausse, IC 95 % au-dessus de 0 | pas de hausse |
| Jeu principal, Recall@5 (dense) | pas de baisse significative | baisse (oubli) |
| Mémoire GPU | < 15 Go sur T4 | OOM → voir la cellule 6 |

**Avant de lancer** : `python -m rag.finetune export` sur votre PC, puis déposez
`data/finetune/colab_bundle.zip` dans **Mon Drive/ifrs17_copilot/finetune/**.
Exécution → Modifier le type d'exécution → **GPU T4**.

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Pas de GPU : Exécution → Modifier le type d'exécution → T4"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Drive et données
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, json
BASE = '/content/drive/MyDrive/ifrs17_copilot'
BUNDLE = f'{BASE}/finetune/colab_bundle.zip'      # ou colab_bundle_holdout.zip (variante stricte)
OUT = f'{BASE}/models/bge-m3-ifrs17'
CKPT = f'{BASE}/checkpoints/bge-m3-ifrs17'
assert os.path.exists(BUNDLE), f"Déposez le paquet ici : {BUNDLE}"
os.makedirs('/content/ft', exist_ok=True)
zipfile.ZipFile(BUNDLE).extractall('/content/ft')
print(sorted(os.listdir('/content/ft')))

In [ ]:
# 3. Bibliothèques (torch est déjà installé sur Colab)
!pip install -q -U "sentence-transformers>=3.3" datasets
import sentence_transformers
print(sentence_transformers.__version__)

In [ ]:
# 4. Évaluation : mêmes métriques que eval/eval_retrieval.py (mode dense, découpage B)
import numpy as np

def jsonl(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

chunks = jsonl('/content/ft/chunks_B.jsonl')
DOCS = [c['embed_text'] for c in chunks]
PIDS = [set(c['paragraph_ids']) for c in chunks]
SETS = {}
for name, f in [('quiz du cours', 'course_quiz.jsonl'), ('jeu principal', 'rag_questions.jsonl')]:
    qs = [q for q in jsonl(f'/content/ft/{f}') if q.get('gold')]   # les questions « piège » n'ont pas de gold
    SETS[name] = qs
    print(f"{name} : {len(qs)} questions")

def recall_at(ranked, gold, k):
    found = set().union(*(PIDS[i] for i in ranked[:k]))
    return len(found & gold) / len(gold)

def mrr(ranked, gold, k=10):
    for r, i in enumerate(ranked[:k], start=1):
        if PIDS[i] & gold:
            return 1 / r
    return 0.0

def evaluate(model):
    doc_vecs = model.encode(DOCS, batch_size=16, normalize_embeddings=True, show_progress_bar=False)
    out = {}
    for name, qs in SETS.items():
        q_vecs = model.encode([q['question'] for q in qs], normalize_embeddings=True)
        per_q = []
        for q, v in zip(qs, q_vecs):
            ranked = list(np.argsort(-doc_vecs @ v)[:10])
            gold = set(q['gold'])
            per_q.append({'id': q['id'], 'r@1': recall_at(ranked, gold, 1), 'r@5': recall_at(ranked, gold, 5),
                          'r@10': recall_at(ranked, gold, 10), 'mrr': mrr(ranked, gold)})
        out[name] = per_q
    return out

def summary(res):
    return {name: {m: round(float(np.mean([r[m] for r in rows])), 3) for m in ('r@1', 'r@5', 'r@10', 'mrr')}
            for name, rows in res.items()}

def bootstrap_diff(a, b, n=10_000, seed=0):
    # écart moyen apparié (b − a) et intervalle de confiance à 95 %, questions rééchantillonnées
    a, b = np.array(a), np.array(b)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n, len(a)))
    d = (b[idx] - a[idx]).mean(axis=1)
    return float((b - a).mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

In [ ]:
# 5. Référence : BGE-M3 d'origine
from sentence_transformers import SentenceTransformer
base = SentenceTransformer('BAAI/bge-m3', device='cuda')
base.half()                                   # évaluation en FP16 (T4)
MAX_LEN = base.max_seq_length                 # 8192 : remis tel quel après l'entraînement
before = evaluate(base)
print(json.dumps(summary(before), ensure_ascii=False, indent=1))
del base; torch.cuda.empty_cache()

In [ ]:
# 6. Entraînement
# En cas de manque de mémoire (OOM) : MINI_BATCH = 4, puis TRAIN_LEN = 256, puis OPTIM = 'adafactor'.
from datasets import load_dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss   # v3 à v6

EPOCHS, BATCH, MINI_BATCH, LR, TRAIN_LEN, OPTIM = 2, 64, 8, 1e-5, 384, 'adamw_torch'

train = load_dataset('json', data_files='/content/ft/train.jsonl', split='train')
print(train)
model = SentenceTransformer('BAAI/bge-m3', device='cuda')
model.max_seq_length = TRAIN_LEN              # ~92 % des passages tiennent en entier (estimation : 1,4 token par mot)
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_BATCH)
args = SentenceTransformerTrainingArguments(
    output_dir=CKPT, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
    learning_rate=LR, warmup_ratio=0.1, fp16=True, optim=OPTIM,
    batch_sampler='no_duplicates',        # même passage deux fois dans un lot = faux négatif
    save_strategy='epoch', save_total_limit=1, logging_steps=5, report_to='none', seed=42,
)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train, loss=loss)
torch.cuda.reset_peak_memory_stats()
resume = os.path.isdir(CKPT) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT))
trainer.train(resume_from_checkpoint=resume or None)   # reprise après une coupure de Colab
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"mémoire GPU maximale : {peak_gb:.1f} Go")

In [ ]:
# 7. Évaluation du modèle adapté, écarts avec intervalle de confiance
model.max_seq_length = MAX_LEN
model.half()
after = evaluate(model)
report = {'avant': summary(before), 'apres': summary(after), 'ecarts': {},
          'parametres': dict(epochs=EPOCHS, batch=BATCH, lr=LR, train_len=TRAIN_LEN, paires=len(train),
                             paquet=os.path.basename(BUNDLE), memoire_gpu_go=round(peak_gb, 1))}
for name in SETS:
    for m in ('r@5', 'mrr'):
        d = bootstrap_diff([r[m] for r in before[name]], [r[m] for r in after[name]])
        report['ecarts'][f'{name} · {m}'] = [round(x, 3) for x in d]
print(json.dumps(report, ensure_ascii=False, indent=1))

print('\n| Jeu | Recall@5 avant | après | écart [IC 95 %] | MRR avant | après |')
print('| --- | --- | --- | --- | --- | --- |')
for name in SETS:
    a, b, e = report['avant'][name], report['apres'][name], report['ecarts'][f'{name} · r@5']
    print(f"| {name} | {a['r@5']:.3f} | {b['r@5']:.3f} | {e[0]:+.3f} [{e[1]:+.3f} ; {e[2]:+.3f}] | {a['mrr']:.3f} | {b['mrr']:.3f} |")

# questions du quiz gagnées et perdues : à lire avant de conclure
for b_, a_ in zip(before['quiz du cours'], after['quiz du cours']):
    if b_['r@5'] != a_['r@5']:
        print(f"  {b_['id']} : {b_['r@5']:.2f} → {a_['r@5']:.2f}")

In [ ]:
# 8. Sauvegarde sur Drive (FP16 : ~1,1 Go au lieu de 2,2 Go)
os.makedirs(OUT, exist_ok=True)
model.save(OUT)
json.dump(report, open(f'{OUT}/finetune_report.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(f"modèle enregistré dans {OUT}")
print("Récupération : drive.google.com → Mon Drive/ifrs17_copilot/models → clic droit sur "
      "bge-m3-ifrs17 → Télécharger, puis décompresser dans le dossier models/ du projet.")

## Ensuite, sur votre PC

```powershell
# le dossier models\bge-m3-ifrs17 doit contenir config.json, model.safetensors, modules.json…
python -m eval.eval_retrieval --model BAAI/bge-m3 --model models/bge-m3-ifrs17 --questions eval/course_quiz.jsonl
python -m eval.eval_retrieval --model BAAI/bge-m3 --model models/bge-m3-ifrs17
```

Ces deux commandes mesurent aussi le mode **hybride + glossaire** (celui de l'application), que ce
notebook ne mesure pas. On ne garde le modèle que si les critères go / no-go du début sont remplis
dans ce mode-là ; sinon, la décision D-046 est notée « rejetée (mesurée) », comme D-045.